# SDR Trade Flow Analysis

## Comprehensive Trade Flow Analytics for US Interest Rate Swaps

This notebook provides in-depth analysis of SDR (Swap Data Repository) trade flow data for a US interest rate swaps trading desk. It covers:

1. **Volume Analysis** - Trade counts and notional volumes by time period
2. **Product Distribution** - Breakdown by swap type (OIS, Fixed-Float, Basis)
3. **Tenor Bucketing** - Distribution of trades by maturity
4. **Venue Analysis** - SEF vs Off-Facility trading
5. **Block Trade Analytics** - Block vs non-block trade analysis
6. **Clearing Status** - Cleared vs uncleared breakdown

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

# Initialize the SDR Data Builder with cache path
cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

## 1. Data Loading and Preprocessing

In [ ]:
# Define the time range for analysis
# Modify these dates as needed for your analysis period
start = NY_tz.localize(datetime.datetime(2025, 12, 19, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 19, 17, 0))

print(f"Fetching SDR data from {start} to {end}")

In [ ]:
# Fetch SDR trades
raw_df = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES"
)

print(f"Total trades fetched: {len(raw_df):,}")
raw_df.head()

In [ ]:
def preprocess_sdr_data(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess SDR data for analysis."""
    df = df.copy()
    
    # Filter to new trades only (exclude amendments, corrections, cancellations)
    df = df[df['Action type'] == 'NEWT'].copy()
    
    # Parse notional amounts (remove commas and convert to float)
    for col in ['Notional amount-Leg 1', 'Notional amount-Leg 2']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(',', '').str.replace(' ', '')
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Parse fixed rate
    df['Fixed rate-Leg 1'] = pd.to_numeric(df['Fixed rate-Leg 1'], errors='coerce')
    
    # Calculate tenor in years
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    
    # Create tenor buckets
    def assign_tenor_bucket(years):
        if pd.isna(years) or years <= 0:
            return 'Unknown'
        elif years <= 0.25:
            return '0-3M'
        elif years <= 0.5:
            return '3-6M'
        elif years <= 1:
            return '6M-1Y'
        elif years <= 2:
            return '1-2Y'
        elif years <= 3:
            return '2-3Y'
        elif years <= 5:
            return '3-5Y'
        elif years <= 7:
            return '5-7Y'
        elif years <= 10:
            return '7-10Y'
        elif years <= 15:
            return '10-15Y'
        elif years <= 20:
            return '15-20Y'
        elif years <= 30:
            return '20-30Y'
        else:
            return '30Y+'
    
    df['Tenor_Bucket'] = df['Tenor_Years'].apply(assign_tenor_bucket)
    
    # Extract product type from UPI FISN
    def classify_product(row):
        fisn = str(row.get('UPI FISN', '')).upper()
        underlier = str(row.get('UPI Underlier Name', '')).upper()
        
        if 'OIS' in fisn or 'COMPOUND' in underlier or 'SOFR-OIS' in underlier:
            return 'OIS'
        elif 'FXD FLT' in fisn or 'FIXED' in fisn:
            return 'Fixed-Float'
        elif 'BASIS' in fisn:
            return 'Basis'
        elif 'SWAPTION' in fisn or 'CALL' in fisn or 'PUT' in fisn:
            return 'Swaption'
        elif 'CAP' in fisn or 'FLOOR' in fisn:
            return 'Cap/Floor'
        elif 'FRA' in fisn:
            return 'FRA'
        else:
            return 'Other'
    
    df['Product_Type'] = df.apply(classify_product, axis=1)
    
    # Classify venue
    def classify_venue(platform):
        platform = str(platform).upper()
        if platform in ['XOFF', 'OFF', 'NaN', '', 'NONE']:
            return 'Off-Facility'
        else:
            return 'SEF'
    
    df['Venue_Type'] = df['Platform identifier'].apply(classify_venue)
    
    # Block trade indicator
    df['Is_Block'] = df['Block trade election indicator'].astype(str).str.upper() == 'TRUE'
    
    # Cleared status
    df['Is_Cleared'] = df['Cleared'].isin(['Y', 'C', 'I'])
    
    # Convert event timestamp to NY timezone for analysis
    df['Event timestamp'] = pd.to_datetime(df['Event timestamp'], utc=True)
    df['Execution Timestamp'] = pd.to_datetime(df['Execution Timestamp'], utc=True)
    df['Event_Time_NY'] = df['Event timestamp'].dt.tz_convert('America/New_York')
    df['Execution_Time_NY'] = df['Execution Timestamp'].dt.tz_convert('America/New_York')
    df['Hour_NY'] = df['Event_Time_NY'].dt.hour
    
    return df

df = preprocess_sdr_data(raw_df)
print(f"Processed trades (new trades only): {len(df):,}")

## 2. Volume Analysis

In [ ]:
# Filter to USD SOFR swaps for primary analysis
usd_mask = (df['Notional currency-Leg 1'] == 'USD') & (df['UPI Underlier Name'].str.contains('SOFR', case=False, na=False))
usd_sofr_df = df[usd_mask].copy()

print(f"USD SOFR trades: {len(usd_sofr_df):,}")
print(f"Total notional (Leg 1): ${usd_sofr_df['Notional amount-Leg 1'].sum():,.0f}")

In [ ]:
# Hourly trade volume
hourly_counts = usd_sofr_df.groupby('Hour_NY').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
hourly_counts.columns = ['Hour', 'Trade_Count', 'Total_Notional']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Trade Count by Hour (NY Time)', 'Total Notional by Hour (NY Time)'])

fig.add_trace(
    go.Bar(x=hourly_counts['Hour'], y=hourly_counts['Trade_Count'], name='Trade Count', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=hourly_counts['Hour'], y=hourly_counts['Total_Notional']/1e9, name='Notional ($B)', marker_color='darkgreen'),
    row=2, col=1
)

fig.update_layout(height=700, title_text='USD SOFR Swap Trading Activity by Hour',
                  showlegend=False)
fig.update_xaxes(title_text='Hour (NY Time)', row=2, col=1)
fig.update_yaxes(title_text='Trade Count', row=1, col=1)
fig.update_yaxes(title_text='Notional ($B)', row=2, col=1)
fig.show()

In [ ]:
# Trade count by 15-minute intervals
usd_sofr_df['Time_Bucket_15min'] = usd_sofr_df['Event_Time_NY'].dt.floor('15min')
time_series = usd_sofr_df.groupby('Time_Bucket_15min').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
time_series.columns = ['Time', 'Trade_Count', 'Total_Notional']

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_series['Time'], y=time_series['Trade_Count'],
                         mode='lines+markers', name='Trade Count',
                         line=dict(color='steelblue', width=2)))
fig.update_layout(title='USD SOFR Trade Count (15-min intervals)',
                  xaxis_title='Time (NY)', yaxis_title='Trade Count',
                  height=500)
fig.show()

## 3. Product Distribution

In [ ]:
# Product type distribution
product_dist = usd_sofr_df.groupby('Product_Type').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
product_dist.columns = ['Product_Type', 'Trade_Count', 'Total_Notional']
product_dist['Notional_B'] = product_dist['Total_Notional'] / 1e9
product_dist = product_dist.sort_values('Trade_Count', ascending=True)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
                    subplot_titles=['Trade Count Distribution', 'Notional Distribution'])

fig.add_trace(
    go.Pie(labels=product_dist['Product_Type'], values=product_dist['Trade_Count'],
           textinfo='label+percent', hole=0.4),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=product_dist['Product_Type'], values=product_dist['Notional_B'],
           textinfo='label+percent', hole=0.4),
    row=1, col=2
)

fig.update_layout(title_text='USD SOFR Product Distribution', height=500)
fig.show()

In [ ]:
# Product summary table
product_summary = usd_sofr_df.groupby('Product_Type').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean', 'median', 'min', 'max']
}).round(0)
product_summary.columns = ['Trade_Count', 'Total_Notional', 'Avg_Notional', 'Median_Notional', 'Min_Notional', 'Max_Notional']
product_summary = product_summary.sort_values('Trade_Count', ascending=False)
product_summary

## 4. Tenor Bucketing Analysis

In [ ]:
# Define tenor bucket order
tenor_order = ['0-3M', '3-6M', '6M-1Y', '1-2Y', '2-3Y', '3-5Y', '5-7Y', '7-10Y', '10-15Y', '15-20Y', '20-30Y', '30Y+', 'Unknown']

# Tenor distribution
tenor_dist = usd_sofr_df.groupby('Tenor_Bucket').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
tenor_dist.columns = ['Tenor_Bucket', 'Trade_Count', 'Total_Notional']
tenor_dist['Notional_B'] = tenor_dist['Total_Notional'] / 1e9
tenor_dist['Tenor_Bucket'] = pd.Categorical(tenor_dist['Tenor_Bucket'], categories=tenor_order, ordered=True)
tenor_dist = tenor_dist.sort_values('Tenor_Bucket')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Trade Count by Tenor', 'Notional ($B) by Tenor'])

fig.add_trace(
    go.Bar(x=tenor_dist['Tenor_Bucket'].astype(str), y=tenor_dist['Trade_Count'],
           name='Trade Count', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=tenor_dist['Tenor_Bucket'].astype(str), y=tenor_dist['Notional_B'],
           name='Notional ($B)', marker_color='darkgreen'),
    row=2, col=1
)

fig.update_layout(height=700, title_text='USD SOFR Swap Tenor Distribution', showlegend=False)
fig.show()

In [ ]:
# Heatmap of tenor vs hour
tenor_hour = pd.crosstab(usd_sofr_df['Tenor_Bucket'], usd_sofr_df['Hour_NY'])
# Reorder rows
tenor_hour = tenor_hour.reindex([t for t in tenor_order if t in tenor_hour.index])

fig = go.Figure(data=go.Heatmap(
    z=tenor_hour.values,
    x=tenor_hour.columns,
    y=tenor_hour.index,
    colorscale='Blues',
    hoverongaps=False
))

fig.update_layout(
    title='Trade Activity Heatmap: Tenor vs Hour (NY Time)',
    xaxis_title='Hour (NY Time)',
    yaxis_title='Tenor Bucket',
    height=600
)
fig.show()

## 5. Venue Analysis

In [ ]:
# SEF vs Off-Facility distribution
venue_dist = usd_sofr_df.groupby('Venue_Type').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
venue_dist.columns = ['Venue_Type', 'Trade_Count', 'Total_Notional']

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
                    subplot_titles=['Trade Count by Venue', 'Notional by Venue'])

fig.add_trace(
    go.Pie(labels=venue_dist['Venue_Type'], values=venue_dist['Trade_Count'],
           textinfo='label+percent', hole=0.4,
           marker_colors=['steelblue', 'lightgray']),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=venue_dist['Venue_Type'], values=venue_dist['Total_Notional'],
           textinfo='label+percent', hole=0.4,
           marker_colors=['darkgreen', 'lightgray']),
    row=1, col=2
)

fig.update_layout(title_text='SEF vs Off-Facility Trading', height=400)
fig.show()

In [ ]:
# Platform breakdown
sef_trades = usd_sofr_df[usd_sofr_df['Venue_Type'] == 'SEF']
platform_dist = sef_trades.groupby('Platform identifier').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
platform_dist.columns = ['Platform', 'Trade_Count', 'Total_Notional']
platform_dist['Notional_B'] = platform_dist['Total_Notional'] / 1e9
platform_dist = platform_dist.sort_values('Trade_Count', ascending=False).head(10)

fig = go.Figure()
fig.add_trace(go.Bar(x=platform_dist['Platform'], y=platform_dist['Trade_Count'],
                     name='Trade Count', marker_color='steelblue'))
fig.update_layout(title='Top 10 SEF Platforms by Trade Count',
                  xaxis_title='Platform', yaxis_title='Trade Count',
                  height=500)
fig.show()

## 6. Block Trade Analysis

In [ ]:
# Block vs non-block trades
block_dist = usd_sofr_df.groupby('Is_Block').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': ['sum', 'mean', 'median']
}).reset_index()
block_dist.columns = ['Is_Block', 'Trade_Count', 'Total_Notional', 'Avg_Notional', 'Median_Notional']
block_dist['Is_Block'] = block_dist['Is_Block'].map({True: 'Block Trade', False: 'Non-Block'})

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'bar'}]],
                    subplot_titles=['Trade Count Distribution', 'Notional Comparison'])

fig.add_trace(
    go.Pie(labels=block_dist['Is_Block'], values=block_dist['Trade_Count'],
           textinfo='label+percent', hole=0.4),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=block_dist['Is_Block'], y=block_dist['Total_Notional']/1e9,
           name='Total Notional ($B)', marker_color='steelblue'),
    row=1, col=2
)

fig.update_layout(title_text='Block Trade Analysis', height=400)
fig.show()

In [ ]:
# Block trade size distribution by tenor
block_trades = usd_sofr_df[usd_sofr_df['Is_Block']]
if len(block_trades) > 0:
    block_tenor = block_trades.groupby('Tenor_Bucket').agg({
        'Dissemination Identifier': 'count',
        'Notional amount-Leg 1': ['sum', 'mean']
    }).reset_index()
    block_tenor.columns = ['Tenor_Bucket', 'Trade_Count', 'Total_Notional', 'Avg_Notional']
    block_tenor['Tenor_Bucket'] = pd.Categorical(block_tenor['Tenor_Bucket'], categories=tenor_order, ordered=True)
    block_tenor = block_tenor.sort_values('Tenor_Bucket')
    
    fig = go.Figure()
    fig.add_trace(go.Bar(x=block_tenor['Tenor_Bucket'].astype(str), 
                         y=block_tenor['Avg_Notional']/1e6,
                         name='Avg Notional ($M)', marker_color='darkred'))
    fig.update_layout(title='Average Block Trade Size by Tenor',
                      xaxis_title='Tenor', yaxis_title='Avg Notional ($M)',
                      height=500)
    fig.show()
else:
    print("No block trades found in the dataset.")

## 7. Clearing Status Analysis

In [ ]:
# Cleared vs uncleared
clearing_dist = usd_sofr_df.groupby('Cleared').agg({
    'Dissemination Identifier': 'count',
    'Notional amount-Leg 1': 'sum'
}).reset_index()
clearing_dist.columns = ['Cleared', 'Trade_Count', 'Total_Notional']

# Map clearing codes
clearing_map = {
    'Y': 'Cleared',
    'N': 'Not Cleared',
    'C': 'Cleared',
    'I': 'Intent to Clear'
}
clearing_dist['Clearing_Status'] = clearing_dist['Cleared'].map(clearing_map).fillna('Unknown')

fig = go.Figure()
fig.add_trace(go.Bar(x=clearing_dist['Clearing_Status'], y=clearing_dist['Trade_Count'],
                     text=clearing_dist['Trade_Count'],
                     textposition='auto',
                     marker_color='steelblue'))
fig.update_layout(title='Trade Count by Clearing Status',
                  xaxis_title='Clearing Status', yaxis_title='Trade Count',
                  height=400)
fig.show()

## 8. Summary Statistics

In [ ]:
# Overall summary
print("=" * 60)
print("USD SOFR SWAP TRADING SUMMARY")
print("=" * 60)
print(f"Analysis Period: {start.strftime('%Y-%m-%d %H:%M')} to {end.strftime('%Y-%m-%d %H:%M')} NY")
print(f"\nTotal Trades: {len(usd_sofr_df):,}")
print(f"Total Notional: ${usd_sofr_df['Notional amount-Leg 1'].sum():,.0f}")
print(f"Average Trade Size: ${usd_sofr_df['Notional amount-Leg 1'].mean():,.0f}")
print(f"Median Trade Size: ${usd_sofr_df['Notional amount-Leg 1'].median():,.0f}")
print(f"\nSEF Trading: {len(usd_sofr_df[usd_sofr_df['Venue_Type'] == 'SEF']):,} trades ({len(usd_sofr_df[usd_sofr_df['Venue_Type'] == 'SEF'])/len(usd_sofr_df)*100:.1f}%)")
print(f"Block Trades: {len(usd_sofr_df[usd_sofr_df['Is_Block']]):,} trades ({len(usd_sofr_df[usd_sofr_df['Is_Block']])/len(usd_sofr_df)*100:.1f}%)")
print(f"Cleared Trades: {len(usd_sofr_df[usd_sofr_df['Is_Cleared']]):,} trades ({len(usd_sofr_df[usd_sofr_df['Is_Cleared']])/len(usd_sofr_df)*100:.1f}%)")

In [ ]:
# Top tenors by volume
print("\nTop Tenors by Trade Count:")
top_tenors = usd_sofr_df.groupby('Tenor_Bucket')['Dissemination Identifier'].count().sort_values(ascending=False).head(5)
for tenor, count in top_tenors.items():
    print(f"  {tenor}: {count:,} trades")